# Autodiff from Scratch

## Finite differences, forward mode, reverse mode, and a tiny neural network

Automatic differentiation (AD) is the machinery behind `loss.backward()`, but it is not a numerical approximation and it is not symbolic algebra. In this notebook you will build both major modes of AD, test them against calculus, and use a scalar reverse-mode engine to train an XOR network.

**Level:** undergraduate prerequisite deep dive  
**Time:** 90–120 minutes, including the investigations  
**Dependencies:** NumPy and Matplotlib only; CPU execution is deterministic

### Learning goals

By the end, you should be able to:

1. explain why finite differences are useful for checking derivatives but unsuitable as a training engine;
2. compute a Jacobian-vector product (JVP) with dual numbers;
3. trace a scalar computation graph and propagate adjoints in reverse topological order;
4. explain why gradients from fan-out paths must accumulate with `+=`;
5. recover Jacobians, vector-Jacobian products (VJPs), and Hessian-vector products (HVPs); and
6. debug and train a small multilayer perceptron using only the engine built here.

### Where this fits

Read the wiki's [Matrix Calculus and Automatic Differentiation](https://github.com/jjames/llm-wiki/blob/main/lessons/prerequisites/02-calculus-optimization/p4-matrix-calculus-autodiff.md) module first. This notebook supplies the missing implementation experience, then hands off to [Notebook 14](14_calculus_optimization_mastery.ipynb) for optimization and gradient-checking practice.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(17)


## 1. Finite differences: a diagnostic, not autodiff

The central-difference estimate

$$
f'(x) \approx \frac{f(x+h)-f(x-h)}{2h}
$$

has truncation error when $h$ is too large. When $h$ is extremely small, the subtraction loses significant digits and floating-point roundoff dominates. AD instead applies exact local derivative rules to the program's operations; its numerical error comes from ordinary floating-point arithmetic, not a chosen step size.


In [ ]:
def f_scalar(x):
    return x**3 - 2.0 * x + np.exp(0.5 * x)


def df_scalar(x):
    return 3.0 * x**2 - 2.0 + 0.5 * np.exp(0.5 * x)


def central_difference(f, x, h):
    return (f(x + h) - f(x - h)) / (2.0 * h)


x0 = 0.7
steps = np.logspace(-1, -16, 16)
estimates = np.array([central_difference(f_scalar, x0, h) for h in steps])
errors = np.abs(estimates - df_scalar(x0))

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(steps, errors, marker="o")
ax.set(xlabel="step size h", ylabel="absolute error", title="Central differences have a useful middle range")
ax.grid(True, which="both", alpha=0.25)
plt.show()

best_index = int(np.argmin(errors))
print(f"analytic derivative: {df_scalar(x0):.12f}")
print(f"best h: {steps[best_index]:.1e}; error: {errors[best_index]:.3e}")

assert errors[best_index] < 1e-8
assert errors[-1] > errors[best_index]


### Investigation 1

- Why does the curve turn upward on the right side of the plot (small $h$)?
- Change `x0`, or replace `f_scalar` with a rapidly varying function. Does the best $h$ stay fixed?
- A gradient of a model with $P$ parameters needs roughly $2P$ function evaluations by central differences. Keep that scaling in mind when reverse mode uses one forward and one backward pass.


## 2. Forward mode with dual numbers

A dual number stores a **primal** value $x$ and a **tangent** $\dot{x}$. Overloading each operation propagates both. For multiplication,

$$
(x,\dot{x})(y,\dot{y})=(xy,\;\dot{x}y+x\dot{y}).
$$

If a function $f:\mathbb{R}^n\to\mathbb{R}^m$ receives input tangents collected in a direction $u$, its output tangents are the JVP $J_f(x)u$. One forward sweep gives one directional derivative, regardless of $m$.


In [ ]:
class Dual:
    """A scalar primal value paired with one directional derivative."""

    def __init__(self, primal, tangent=0.0):
        self.primal = float(primal)
        self.tangent = float(tangent)

    @staticmethod
    def _coerce(other):
        return other if isinstance(other, Dual) else Dual(other)

    def __add__(self, other):
        other = self._coerce(other)
        return Dual(self.primal + other.primal, self.tangent + other.tangent)

    __radd__ = __add__

    def __neg__(self):
        return Dual(-self.primal, -self.tangent)

    def __sub__(self, other):
        return self + (-self._coerce(other))

    def __rsub__(self, other):
        return self._coerce(other) + (-self)

    def __mul__(self, other):
        other = self._coerce(other)
        primal = self.primal * other.primal
        tangent = self.tangent * other.primal + self.primal * other.tangent
        return Dual(primal, tangent)

    __rmul__ = __mul__

    def __pow__(self, exponent):
        if not isinstance(exponent, (int, float)):
            return NotImplemented
        primal = self.primal**exponent
        tangent = exponent * self.primal ** (exponent - 1) * self.tangent
        return Dual(primal, tangent)

    def exp(self):
        primal = math.exp(self.primal)
        return Dual(primal, primal * self.tangent)

    def tanh(self):
        primal = math.tanh(self.primal)
        return Dual(primal, (1.0 - primal**2) * self.tangent)

    def __repr__(self):
        return f"Dual(primal={self.primal:.6g}, tangent={self.tangent:.6g})"


In [ ]:
def dual_vector_function(x, y):
    return x * y + x**2, (x - y).exp()


point = np.array([1.2, -0.4])
direction = np.array([0.3, 0.8])
x_dual = Dual(point[0], direction[0])
y_dual = Dual(point[1], direction[1])
outputs = dual_vector_function(x_dual, y_dual)
jvp_forward = np.array([output.tangent for output in outputs])

x, y = point
analytic_jacobian = np.array([
    [y + 2.0 * x, x],
    [np.exp(x - y), -np.exp(x - y)],
])
jvp_analytic = analytic_jacobian @ direction

print("Jacobian:\n", analytic_jacobian)
print("forward-mode JVP:", jvp_forward)
print("matrix J @ direction:", jvp_analytic)
np.testing.assert_allclose(jvp_forward, jvp_analytic, rtol=1e-12, atol=1e-12)


### Investigation 2

Run the function twice with seed directions $(1,0)$ and $(0,1)$. The two JVPs are the columns of the full Jacobian. This is why forward mode is attractive when there are few inputs and potentially many outputs—and costly when a model has millions of parameters.


## 3. Reverse mode: a scalar computation graph

Reverse mode first evaluates the program and records a directed acyclic graph. It then seeds the scalar output with adjoint $1$ and visits nodes in reverse topological order. Each node applies a local derivative rule and sends contributions to its parents.

For $f:\mathbb{R}^n\to\mathbb{R}$, one reverse sweep computes all $n$ input derivatives. That input-heavy, scalar-output shape is exactly a neural-network loss.


In [ ]:
class Value:
    """A scalar value and the graph needed for reverse-mode autodiff."""

    def __init__(self, data, _children=(), _op="", label=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = tuple(_children)
        self._op = _op
        self.label = label
        self._backward = lambda: None

    @staticmethod
    def _coerce(other):
        return other if isinstance(other, Value) else Value(other)

    def __add__(self, other):
        other = self._coerce(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    __radd__ = __add__

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-self._coerce(other))

    def __rsub__(self, other):
        return self._coerce(other) + (-self)

    def __mul__(self, other):
        other = self._coerce(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    __rmul__ = __mul__

    def __pow__(self, exponent):
        if not isinstance(exponent, (int, float)):
            return NotImplemented
        out = Value(self.data**exponent, (self,), f"**{exponent}")

        def _backward():
            self.grad += exponent * self.data ** (exponent - 1) * out.grad

        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = self._coerce(other)
        return self * other**-1

    def __rtruediv__(self, other):
        return self._coerce(other) * self**-1

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward
        return out

    def log(self):
        if self.data <= 0:
            raise ValueError("log is defined only for positive values")
        out = Value(math.log(self.data), (self,), "log")

        def _backward():
            self.grad += out.grad / self.data

        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1.0 - t**2) * out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "ReLU")

        def _backward():
            self.grad += (self.data > 0.0) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topology(node):
            if node not in visited:
                visited.add(node)
                for parent in node._prev:
                    build_topology(parent)
                topo.append(node)

        build_topology(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
        return topo

    def __repr__(self):
        name = f"{self.label}: " if self.label else ""
        return f"Value({name}data={self.data:.6g}, grad={self.grad:.6g})"


Three details carry most of the correctness burden:

1. `_prev` records graph edges during the forward pass.
2. A topological ordering ensures every child's adjoint is available before a node propagates backward.
3. Every local rule uses `+=`, because the same value may influence the output along several paths.

Like PyTorch, this engine accumulates gradients. Use fresh graphs for independent examples, and explicitly zero parameter gradients before each training step.


In [ ]:
x = Value(0.7, label="x")
y = Value(-0.2, label="y")
loss = (x * y).exp() + x
topology = loss.backward()

expected_dx = y.data * math.exp(x.data * y.data) + 1.0
expected_dy = x.data * math.exp(x.data * y.data)

print(f"loss = {loss.data:.8f}")
print(f"dL/dx = {x.grad:.8f}; analytic = {expected_dx:.8f}")
print(f"dL/dy = {y.grad:.8f}; analytic = {expected_dy:.8f}")
print(f"graph nodes in topological order: {len(topology)}")

np.testing.assert_allclose([x.grad, y.grad], [expected_dx, expected_dy], rtol=1e-12)


In [ ]:
def graph_table(output):
    """Return a compact, deterministic description of a graph."""
    nodes = []
    visited = set()

    def visit(node):
        if node in visited:
            return
        visited.add(node)
        for parent in node._prev:
            visit(parent)
        nodes.append(node)

    visit(output)
    index = {node: i for i, node in enumerate(nodes)}
    return [
        {
            "node": index[node],
            "label/op": node.label or node._op or "leaf",
            "data": round(node.data, 6),
            "grad": round(node.grad, 6),
            "parents": [index[parent] for parent in node._prev],
        }
        for node in nodes
    ]


for row in graph_table(loss):
    print(row)


## 4. Fan-out and gradient accumulation

If $z=x^2+x$, then $x$ reaches $z$ through three graph edges: twice through multiplication and once through addition. Its adjoint is the **sum** of those path contributions, $2x+1$. Assignment (`=`) would silently keep only the last path; accumulation (`+=`) implements the multivariable chain rule.


In [ ]:
x = Value(3.0, label="shared x")
z = x * x + x
z.backward()

print(f"autodiff dz/dx = {x.grad}; analytic = {2 * x.data + 1}")
assert x.grad == 7.0


## 5. Jacobians and VJPs from a scalar engine

For vector output $f(x)$, reverse mode naturally accepts an output cotangent $v$ and returns

$$
v^\top J_f(x),
$$

usually called a VJP. We can form the scalar $v^\top f(x)$ and call `backward`. To materialize a full $m\times n$ Jacobian, use each output basis vector in turn (here we rebuild the graph to avoid stale accumulated gradients).


In [ ]:
def value_vector_function(x_data, y_data):
    x = Value(x_data, label="x")
    y = Value(y_data, label="y")
    outputs = (x * y + x**2, (x - y).exp())
    return x, y, outputs


def reverse_jacobian(x_data, y_data):
    rows = []
    for output_index in range(2):
        x, y, outputs = value_vector_function(x_data, y_data)
        outputs[output_index].backward()
        rows.append([x.grad, y.grad])
    return np.array(rows)


jacobian_reverse = reverse_jacobian(*point)
cotangent = np.array([0.25, -0.6])
x, y, outputs = value_vector_function(*point)
weighted_output = cotangent[0] * outputs[0] + cotangent[1] * outputs[1]
weighted_output.backward()
vjp_reverse = np.array([x.grad, y.grad])

print("reverse-mode Jacobian:\n", jacobian_reverse)
print("reverse-mode VJP:", vjp_reverse)
print("matrix v @ J:", cotangent @ analytic_jacobian)

np.testing.assert_allclose(jacobian_reverse, analytic_jacobian, rtol=1e-12, atol=1e-12)
np.testing.assert_allclose(vjp_reverse, cotangent @ analytic_jacobian, rtol=1e-12, atol=1e-12)


### Mode selection by shape

| Goal | Natural operation | Approximate sweep count |
|---|---|---:|
| One input direction, many outputs | JVP (forward mode) | 1 |
| Many inputs, one scalar loss | VJP/gradient (reverse mode) | 1 |
| Full $m\times n$ Jacobian | $n$ JVPs or $m$ VJPs | $\min(n,m)$ with the better mode |

Frameworks rarely build a huge Jacobian explicitly. They compose JVPs and VJPs with vectors, which is both cheaper and more memory efficient.


## 6. Gradient checking

A strong gradient check compares AD with finite differences at several ordinary, non-singular points. Use double precision, central differences, and both absolute and relative tolerances. Avoid ReLU kinks and other nondifferentiable points: two legitimate derivative conventions can disagree there.


In [ ]:
def reverse_gradient(theta):
    x = Value(theta[0], label="x")
    y = Value(theta[1], label="y")
    objective = (x * y).exp() + x**3 - 0.5 * y**2
    objective.backward()
    return np.array([x.grad, y.grad])


def finite_difference_gradient(f, theta, h=1e-5):
    gradient = np.zeros_like(theta, dtype=float)
    for index in range(len(theta)):
        step = np.zeros_like(theta, dtype=float)
        step[index] = h
        gradient[index] = (f(theta + step) - f(theta - step)) / (2.0 * h)
    return gradient


def numeric_objective(theta):
    x, y = theta
    return np.exp(x * y) + x**3 - 0.5 * y**2


theta = np.array([0.4, -0.7])
ad_gradient = reverse_gradient(theta)
fd_gradient = finite_difference_gradient(numeric_objective, theta)
relative_error = np.linalg.norm(ad_gradient - fd_gradient) / max(
    1.0, np.linalg.norm(ad_gradient), np.linalg.norm(fd_gradient)
)

print("AD gradient:", ad_gradient)
print("finite-difference gradient:", fd_gradient)
print(f"relative error: {relative_error:.3e}")
assert relative_error < 1e-9


## 7. Hessian-vector products without forming the Hessian

For scalar $L(\theta)$, an HVP is $H_L(\theta)u$. Exact HVPs can be obtained by composing forward and reverse AD. Our engine stores adjoints as Python floats, so its backward pass is not itself differentiable. We can still verify the idea by central-differencing reverse-mode gradients:

$$
H(\theta)u \approx \frac{\nabla L(\theta+hu)-\nabla L(\theta-hu)}{2h}.
$$

This is a directional check, not a claim that finite differences are the preferred production implementation.


In [ ]:
def reverse_gradient_for_hvp(theta):
    x = Value(theta[0])
    y = Value(theta[1])
    objective = x**3 + x * y + y.exp()
    objective.backward()
    return np.array([x.grad, y.grad])


theta = np.array([0.6, -0.3])
direction = np.array([0.4, -0.8])
h = 1e-5
hvp_directional = (
    reverse_gradient_for_hvp(theta + h * direction)
    - reverse_gradient_for_hvp(theta - h * direction)
) / (2.0 * h)

x, y = theta
analytic_hessian = np.array([[6.0 * x, 1.0], [1.0, np.exp(y)]])
hvp_analytic = analytic_hessian @ direction

print("directional HVP:", hvp_directional)
print("analytic H @ direction:", hvp_analytic)
np.testing.assert_allclose(hvp_directional, hvp_analytic, rtol=1e-8, atol=1e-8)


## 8. Capstone lab: train XOR through your engine

XOR is not linearly separable. A hidden layer must learn a nonlinear representation before the output neuron can separate the classes. The model below uses only `Value` operations; NumPy is used for initialization, data storage, and reporting—not differentiation.

The training loop exposes the framework steps that are usually hidden:

1. build a fresh graph in the forward pass;
2. compute a scalar mean-squared-error loss;
3. zero parameter gradients;
4. run reverse mode; and
5. update parameter data with stochastic gradient descent.


In [ ]:
class Neuron:
    def __init__(self, input_size, rng, activation=True):
        scale = 1.0 / math.sqrt(input_size)
        self.weights = [Value(rng.normal(0.0, scale)) for _ in range(input_size)]
        self.bias = Value(0.0)
        self.activation = activation

    def __call__(self, inputs):
        preactivation = sum((weight * value for weight, value in zip(self.weights, inputs)), self.bias)
        return preactivation.tanh() if self.activation else preactivation

    def parameters(self):
        return self.weights + [self.bias]


class Layer:
    def __init__(self, input_size, output_size, rng, activation=True):
        self.neurons = [Neuron(input_size, rng, activation) for _ in range(output_size)]

    def __call__(self, inputs):
        return [neuron(inputs) for neuron in self.neurons]

    def parameters(self):
        return [parameter for neuron in self.neurons for parameter in neuron.parameters()]


class MLP:
    def __init__(self, input_size, layer_sizes, rng):
        sizes = [input_size, *layer_sizes]
        self.layers = [
            Layer(sizes[index], sizes[index + 1], rng, activation=True)
            for index in range(len(layer_sizes))
        ]

    def __call__(self, inputs):
        values = list(inputs)
        for layer in self.layers:
            values = layer(values)
        return values[0] if len(values) == 1 else values

    def parameters(self):
        return [parameter for layer in self.layers for parameter in layer.parameters()]


In [ ]:
xor_inputs = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
xor_targets = np.array([-1.0, 1.0, 1.0, -1.0])
model = MLP(2, [6, 1], np.random.default_rng(17))

steps = 800
loss_history = []
for step in range(steps):
    predictions = [model(row) for row in xor_inputs]
    loss = sum((prediction - target) ** 2 for prediction, target in zip(predictions, xor_targets)) / len(xor_targets)

    for parameter in model.parameters():
        parameter.grad = 0.0
    loss.backward()

    learning_rate = 0.12 * (1.0 - step / steps) + 0.02
    for parameter in model.parameters():
        parameter.data -= learning_rate * parameter.grad

    loss_history.append(loss.data)

final_predictions = np.array([model(row).data for row in xor_inputs])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(loss_history)
axes[0].set(xlabel="training step", ylabel="mean squared error", title="XOR training")
axes[0].grid(alpha=0.25)
axes[1].bar(range(4), final_predictions, color=["C0" if target < 0 else "C1" for target in xor_targets])
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set(xticks=range(4), xticklabels=["00", "01", "10", "11"], ylim=(-1.1, 1.1), title="Final predictions")
plt.tight_layout()
plt.show()

print(f"parameters: {len(model.parameters())}")
print(f"initial loss: {loss_history[0]:.6f}; final loss: {loss_history[-1]:.6f}")
print("predictions:", np.round(final_predictions, 4))

assert loss_history[-1] < 0.02
np.testing.assert_array_equal(np.sign(final_predictions), xor_targets)


## 9. Debugging checklist

When an autodiff result looks wrong, check these in order:

1. **Local rules:** test each primitive (`+`, `*`, `exp`, `tanh`) against a hand derivative.
2. **Accumulation:** every parent update must use `+=`; repeated inputs and branches expose this bug.
3. **Traversal:** reverse topological order is essential. A parent must wait for all child contributions.
4. **Gradient lifetime:** zero parameter gradients between optimization steps; rebuild or deliberately retain graphs between backward calls.
5. **Numerical check:** central-difference the scalar objective in double precision with a moderate $h$.
6. **Nonsmooth points:** ReLU has no unique derivative at zero. This engine chooses zero there.
7. **Domain and scale:** `log` needs positive inputs; `exp` can overflow; tiny differences can cancel.
8. **Shapes in real systems:** scalar graphs hide tensor broadcasting and reduction rules. Write down every Jacobian shape before generalizing.


## 10. Cumulative exercises

These are ordered from implementation fluency to design judgment.

1. **Primitive audit.** Add `sin` to `Dual` and `Value`. Check it at five random points, including inside a composite expression.
2. **Recover a Jacobian.** Use forward-mode basis seeds to reconstruct the Jacobian from Section 2. Compare its operation count with the two reverse sweeps in Section 5.
3. **Break accumulation deliberately.** Replace one `+=` with `=` in multiplication. Identify examples that still pass and construct the smallest graph that fails.
4. **Gradient-lifetime diagnostic.** Remove the zeroing loop from XOR training. Record what happens, explain why, and restore the invariant.
5. **Binary cross-entropy.** Add a sigmoid or use a stable logistic-loss expression. Train XOR with targets $0/1$ and compare gradients with mean squared error.
6. **Mini-batches and regularization.** Add an $L_2$ penalty and train on randomly sampled examples. Explain whether the regularizer belongs in the graph.
7. **Exact HVP design.** Sketch the changes required to make backward computations produce `Value` objects rather than floats. Which graph would a second reverse pass traverse?
8. **Oral mastery check.** Without code, explain when you would choose forward mode, reverse mode, a full Jacobian, or finite differences.

### Completion standard

You are ready to move on when you can derive two local backward rules, explain reverse topological traversal, diagnose fan-out accumulation, and implement one new primitive with a passing gradient check.


## References and next steps

- Baydin, Pearlmutter, Radul, and Siskind, [*Automatic Differentiation in Machine Learning: a Survey*](https://arxiv.org/abs/1502.05767).
- Parr and Howard, [*The Matrix Calculus You Need For Deep Learning*](https://arxiv.org/abs/1802.01528).
- JAX documentation, [*The Autodiff Cookbook*](https://docs.jax.dev/en/latest/notebooks/autodiff_cookbook.html) (JVPs, VJPs, Jacobians, and HVPs in a production system).
- Karpathy, [micrograd](https://github.com/karpathy/micrograd), a compact scalar reverse-mode engine that inspired this pedagogical implementation.
- Continue with [Notebook 14: Calculus and Optimization Mastery](14_calculus_optimization_mastery.ipynb), then the calculus/optimization capstone in the wiki's mastery layer.

The central lesson is structural: autodiff is the chain rule organized as a program transformation. Forward mode propagates tangents with the computation; reverse mode records the computation and propagates adjoints against it.
